In [1]:
import ctypes
import random
import re
import time
from datetime import datetime
from os import sep
from sys import platform, path

import numpy as np
import pandas as pd
import serial
import serial.tools.list_ports
import matplotlib.pyplot as plt

plt.style.use("dark_background")


In [2]:
# Load the WaveForms dynamic library + constants (OS-specific path).
if platform.startswith("win"):
    dwf = ctypes.cdll.dwf
    constants_path = "C:" + sep + "Program Files (x86)" + sep + "Digilent" + \
                     sep + "WaveFormsSDK" + sep + "samples" + sep + "py"
elif platform.startswith("darwin"):
    dwf = ctypes.cdll.LoadLibrary(sep + "Library" + sep + "Frameworks" +
                                  sep + "dwf.framework" + sep + "dwf")
    constants_path = (sep + "Applications" + sep + "WaveForms.app" + sep + "Contents" +
                      sep + "Resources" + sep + "SDK" + sep + "samples" + sep + "py")
else:
    dwf = ctypes.cdll.LoadLibrary("libdwf.so")
    constants_path = sep + "usr" + sep + "share" + sep + "digilent" + sep + "waveforms" + \
                     sep + "samples" + sep + "py"
path.append(constants_path)
import dwfconstants as constants


class data:
    handle = ctypes.c_int(0)
    name = ""


def open_wv():
    device_handle = ctypes.c_int()
    dwf.FDwfDeviceOpen(ctypes.c_int(-1), ctypes.byref(device_handle))
    data.handle = device_handle
    return data


def _switch_variable_(device_data, master_state, positive_state, negative_state,
                      positive_voltage, negative_voltage):
    positive_voltage = max(0, min(5, positive_voltage))
    dwf.FDwfAnalogIOChannelNodeSet(device_data.handle, ctypes.c_int(0), ctypes.c_int(1),
                                   ctypes.c_double(positive_voltage))
    negative_voltage = max(-5, min(0, negative_voltage))
    dwf.FDwfAnalogIOChannelNodeSet(device_data.handle, ctypes.c_int(1), ctypes.c_int(1),
                                   ctypes.c_double(negative_voltage))
    dwf.FDwfAnalogIOChannelNodeSet(device_data.handle, ctypes.c_int(0), ctypes.c_int(0),
                                   ctypes.c_int(positive_state))
    dwf.FDwfAnalogIOChannelNodeSet(device_data.handle, ctypes.c_int(1), ctypes.c_int(0),
                                   ctypes.c_int(negative_state))
    dwf.FDwfAnalogIOEnableSet(device_data.handle, ctypes.c_int(master_state))


# ---- Voltage-limit guard: keep the VCCS input inside 0..VMAX volts ----
VMAX = 0.300   # 300 mV hard ceiling

def clamp_drive(offset, amplitude, vmax=VMAX):
    """Return (offset, amplitude) so that offset+amplitude<=vmax and offset-amplitude>=0.
    Shrinks amplitude (and, if needed, offset) rather than exceeding the limit."""
    offset = float(min(max(offset, 0.0), vmax))
    amp_max = min(offset, vmax - offset)          # symmetric headroom about the offset
    amplitude = float(min(max(amplitude, 0.0), max(amp_max, 0.0)))
    if amp_max <= 0:
        print(f"[clamp] offset={offset*1e3:.0f} mV leaves no room for modulation; amplitude=0")
    return offset, amplitude


def dc_offset(volt):
    """Hold the LED at a constant level `volt` (clamped to 0..VMAX)."""
    volt = float(min(max(volt, 0.0), VMAX))
    ch, node = ctypes.c_int(0), ctypes.c_int(0)   # W1, carrier
    dwf.FDwfAnalogOutNodeEnableSet(dev.handle, ch, node, ctypes.c_int(1))
    dwf.FDwfAnalogOutNodeFunctionSet(dev.handle, ch, node, constants.funcDC)
    dwf.FDwfAnalogOutNodeOffsetSet(dev.handle, ch, node, ctypes.c_double(volt))
    dwf.FDwfAnalogOutConfigure(dev.handle, ch, ctypes.c_int(1))


def wf_sine(PERIOD, AMPLITUDE, OFFSET):
    """Single sine on W1. PERIOD [s], AMPLITUDE [V peak], OFFSET [V]. Clamped to 0..VMAX."""
    OFFSET, AMPLITUDE = clamp_drive(OFFSET, AMPLITUDE)
    ch, node = ctypes.c_int(0), constants.AnalogOutNodeCarrier
    dwf.FDwfAnalogOutNodeEnableSet(dev.handle, ch, node, ctypes.c_int(1))
    dwf.FDwfAnalogOutNodeFunctionSet(dev.handle, ch, node, constants.funcSine)
    dwf.FDwfAnalogOutNodeFrequencySet(dev.handle, ch, node, ctypes.c_double(1.0 / PERIOD))
    dwf.FDwfAnalogOutNodeAmplitudeSet(dev.handle, ch, node, ctypes.c_double(AMPLITUDE))
    dwf.FDwfAnalogOutNodeOffsetSet(dev.handle, ch, node, ctypes.c_double(OFFSET))
    dwf.FDwfAnalogOutConfigure(dev.handle, ch, ctypes.c_int(1))
    return OFFSET, AMPLITUDE


def wf_custom(samples01, pattern_period_s, AMPLITUDE, OFFSET):
    """Play an arbitrary normalized waveform (multisine / chirp) on W1.

    samples01        : array normalized to peak |value| = 1 (one full pattern).
    pattern_period_s : duration of one pattern repetition [s] (freq = 1/this).
    Output = OFFSET + AMPLITUDE*samples01, clamped so it stays inside 0..VMAX.
    """
    OFFSET, AMPLITUDE = clamp_drive(OFFSET, AMPLITUDE)
    s = np.asarray(samples01, dtype=np.float64)
    peak = np.max(np.abs(s)) or 1.0
    s = s / peak                                  # guarantee unit peak -> amplitude is literal
    buf = (ctypes.c_double * len(s))(*s)
    ch, node = ctypes.c_int(0), constants.AnalogOutNodeCarrier
    dwf.FDwfAnalogOutNodeEnableSet(dev.handle, ch, node, ctypes.c_int(1))
    dwf.FDwfAnalogOutNodeFunctionSet(dev.handle, ch, node, constants.funcCustom)
    dwf.FDwfAnalogOutNodeDataSet(dev.handle, ch, node, buf, ctypes.c_int(len(s)))
    dwf.FDwfAnalogOutNodeFrequencySet(dev.handle, ch, node,
                                      ctypes.c_double(1.0 / pattern_period_s))
    dwf.FDwfAnalogOutNodeAmplitudeSet(dev.handle, ch, node, ctypes.c_double(AMPLITUDE))
    dwf.FDwfAnalogOutNodeOffsetSet(dev.handle, ch, node, ctypes.c_double(OFFSET))
    dwf.FDwfAnalogOutConfigure(dev.handle, ch, ctypes.c_int(1))
    return OFFSET, AMPLITUDE


def wf_park():
    """Stop W1 and drop the LED to 0 V (safe state)."""
    try:
        dc_offset(0.0)
    except Exception as e:
        print("[park] dc_offset(0) failed:", e)
    try:
        dwf.FDwfAnalogOutReset(dev.handle, ctypes.c_int(0))
        dwf.FDwfAnalogOutConfigure(dev.handle, ctypes.c_int(0), ctypes.c_int(0))
    except Exception as e:
        print("[park] W1 reset failed:", e)

In [3]:
# --- open the AD3 and enable the +5V supply for the VCCS ---
dev = open_wv()
if dev.handle.value == 0:
    err = ctypes.create_string_buffer(512)
    dwf.FDwfGetLastErrorMsg(err)
    raise RuntimeError("Waveform generator error: " + err.value.decode())
_switch_variable_(dev, True, True, False, 5.0, 0.0)
dwf.FDwfAnalogIOEnableSet(dev.handle, ctypes.c_int(1))
wf_park()   # start in the safe (0 V) state
print("AD3 open, +5V supply enabled, W1 parked at 0 V.")

AD3 open, +5V supply enabled, W1 parked at 0 V.


In [4]:
import serial
import time

def pwm_serial(value, port='COM20', baudrate=115200, timeout=1):
    """Send 'PWM <value>' to the device without resetting it, and return the reply lines."""
    ser = serial.Serial()
    ser.port     = port
    ser.baudrate = baudrate
    ser.timeout  = timeout
    ser.dtr      = False   # don't drive IO0
    ser.rts      = False   # don't drive EN -> no auto-reset on open
    ser.open()
    try:
        ser.reset_input_buffer()          # clear any stale bytes (device is already running)
        ser.write(f'PWM {value}\r\n'.encode())
        ser.flush()
        time.sleep(0.3)                   # let the CLI produce its reply
        return ser.readlines()
    finally:
        ser.close()


import time

def sleep_interruptible(seconds, step=0.1):
    """Sleep that responds quickly to the Jupyter stop button (KeyboardInterrupt)."""
    end = time.monotonic() + seconds
    try:
        while time.monotonic() < end:
            time.sleep(min(step, end - time.monotonic()))
    except KeyboardInterrupt:
        print(f"Interrupted after ~{seconds - (end - time.monotonic()):.1f}s")
        raise   # remove this line if you'd rather swallow it and continue


In [10]:
pwm_serial(10)

[b'PWM 10\r\n',
 b'W (4409007) ledc: GPIO 4 is not usable, maybe conflict with others\r\n',
 b'PWM GPIO4: 10.00% @ 10000 Hz (12-bit)\r\r\n',
 b'ambyte>  \r\n',
 b'ambyte>  ']

In [9]:
dc_offset(0)

In [6]:
for led_V in [0,0.05,0.1,0.15]:
    dc_offset(led_V)
    for rep in range(5):
        pwm_serial(10)
        time.sleep(60)
        pwm_serial(0)
        time.sleep(5)  # wait a bit before the next repetition
        
    



In [ ]:
for i in range(5):
    try:
        dc_offset(0.1)
        time.sleep(5*60)
        dc_offset(0)
        time.sleep(5*60)
    except KeyboardInterrupt:
        print("Process interrupted by user.")